In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [8]:
base_url = "http://localhost:11434"
model = "llama3.2"

llm = ChatOllama(
    base_url=base_url,
    model=model,
    temperature=0.7)

def word_counter(text):
    return len(text.split())

prompt1 = PromptTemplate(
    template="Generate a 2 to 5 liner joke about {topic}",
    input_variables=['topic']
    )


prompt2 = PromptTemplate(
    template="Generate an explanation about {joke}",
    input_variables=['joke']
    )

parser = StrOutputParser()

sequential_chain = RunnableSequence(prompt1, llm, parser)

parallel_chain = RunnableParallel({
    'joke' : RunnablePassthrough(),
    'explaination' : RunnableSequence(prompt2, llm, parser),
    'word_count' : RunnableLambda(word_counter)
})

# parallel_chain = RunnableParallel({
#     'joke' : RunnablePassthrough(),
#     'explaination' : RunnableSequence(prompt2, llm, parser),
#     'word_count' : RunnableLambda(lambda x: len(x.split())) # in this case no need to create a function before
# })

final_chain = RunnableSequence(sequential_chain, parallel_chain)

results = final_chain.invoke({'topic':'China'})
print(results)

{'joke': 'Why did the Great Wall of China go to therapy? It had a lot of "wall" issues and was feeling a little "blocked." After some self-reflection, it realized it just needed to "build" up its confidence. Now it\'s stronger than ever!', 'explaination': 'I see what you did there! That\'s a clever play on words, using the phrase "wall issues" in a literal sense while also referencing common emotional struggles that people might face. Here\'s an attempt at writing a more serious explanation for why the Great Wall of China might go to therapy:\n\nThe Great Wall of China is one of the most impressive architectural achievements in history, stretching over 13,000 miles across China\'s rugged terrain. However, despite its grandeur and historical significance, the wall has faced numerous challenges and criticisms over the centuries.\n\nIn recent years, the wall has become a symbol of Chinese nationalism and a source of pride for the country. Nevertheless, it also carries a heavy burden of re